In [1]:
# Instala las librerías necesarias:
# exif  -> para leer los metadatos GPS de las imágenes
# gpxpy -> para leer archivos GPX
# folium -> para crear el mapa interactivo
!pip install exif gpxpy folium

# Importa la clase Image desde la librería exif
# Esta clase permite acceder a los metadatos EXIF de una imagen
from exif import Image

# Importa gpxpy para procesar el archivo GPX
# Importa folium para crear mapas interactivos
# Importa base64 para convertir imágenes en texto codificado (necesario para incrustarlas en el mapa)
import gpxpy, folium, base64

# Importa display para poder mostrar el mapa dentro de Google Colab
from IPython.display import display


# Diccionario que relaciona el número del punto en la ruta GPX
# con la ruta del archivo de imagen correspondiente
# 5  -> imagen1.jpg se colocará en el punto 5
# 14 -> imagen2.jpg se colocará en el punto 14
imagenes = {
    5: "/content/drive/MyDrive/Proyecto visor GPX/imagen1.jpg",
    14: "/content/drive/MyDrive/Proyecto visor GPX/imagen2.jpg"
}


# Función que convierte coordenadas en formato
# Grados, Minutos, Segundos (DMS) a formato decimal
def dms_a_decimal(dms, ref):
    # Fórmula matemática de conversión:
    # decimal = grados + (minutos / 60) + (segundos / 3600)
    decimal = dms[0] + dms[1]/60 + dms[2]/3600

    # Si la referencia es Sur (S) o Oeste (W),
    # la coordenada debe ser negativa
    return -decimal if ref in ["S", "W"] else decimal


# Diccionario vacío donde se almacenarán
# las coordenadas decimales de cada imagen
coords_imagenes = {}


# Recorre el diccionario de imágenes
# indice -> número del punto (5 o 14)
# ruta   -> ubicación del archivo de imagen
for indice, ruta in imagenes.items():

    # Abre la imagen en modo binario (rb = read binary)
    # Se usa binario porque las imágenes no son archivos de texto
    with open(ruta, "rb") as img_file:

        # Convierte el archivo abierto en un objeto Image
        # que permite acceder a sus metadatos EXIF
        img = Image(img_file)

    # Verifica que la imagen tenga metadatos EXIF
    # y que tenga atributos de latitud GPS
    if img.has_exif and hasattr(img, "gps_latitude"):

        # Convierte la latitud de DMS a decimal
        lat = dms_a_decimal(img.gps_latitude, img.gps_latitude_ref)

        # Convierte la longitud de DMS a decimal
        lon = dms_a_decimal(img.gps_longitude, img.gps_longitude_ref)

        # Guarda las coordenadas en el diccionario
        # usando como clave el índice (5 o 14)
        coords_imagenes[indice] = (lat, lon)


# Ruta del archivo GPX que contiene la trayectoria
ruta_gpx = "/content/drive/MyDrive/Proyecto visor GPX/Ruta.gpx"


# Abre el archivo GPX en modo lectura
with open(ruta_gpx, "r") as f:

    # Parsea el archivo GPX
    # Esto convierte el archivo en un objeto estructurado
    gpx = gpxpy.parse(f)


# Extrae todas las coordenadas del archivo GPX
# El archivo GPX puede tener:
# tracks -> segmentos -> puntos
# Esta list comprehension recorre toda esa estructura
posiciones = [
    (p.latitude, p.longitude)
    for t in gpx.tracks
    for s in t.segments
    for p in s.points
]


# Crea el mapa centrado en el primer punto de la ruta
# zoom_start=14 define el nivel de acercamiento
m = folium.Map(location=posiciones[0], zoom_start=14)


# Dibuja la línea de la ruta conectando todos los puntos
# color="blue" define el color
# weight=2.5 define el grosor de la línea
folium.PolyLine(posiciones, color="blue", weight=2.5).add_to(m)


# Recorre todas las posiciones y agrega un marcador pequeño
# tipo círculo rojo en cada punto del recorrido
for pos in posiciones:
    folium.CircleMarker(
        pos,
        radius=3,        # tamaño del círculo
        color="red",     # color del borde
        fill=True        # relleno activado
    ).add_to(m)


# Agrega los marcadores especiales con imagen en los puntos 5 y 14
for indice, coord in coords_imagenes.items():

    # Obtiene la ruta de la imagen correspondiente
    ruta_imagen = imagenes[indice]

    # Abre la imagen en modo binario
    with open(ruta_imagen, "rb") as img_file:

        # Convierte la imagen a base64
        # Esto la transforma en texto codificado
        encoded = base64.b64encode(img_file.read()).decode()

    # Crea el contenido HTML del popup
    # Se incrusta la imagen codificada dentro del HTML
    html = f'''
    <h4>Imagen punto {indice}</h4>
    <img src="data:image/jpeg;base64,{encoded}" width="250">
    '''

    # Crea la ventana emergente (popup)
    popup = folium.Popup(html, max_width=300)

    # Crea el marcador tipo Google Maps
    # location -> coordenadas de la imagen
    # popup -> ventana que muestra la imagen
    # icon -> ícono verde con símbolo de cámara
    folium.Marker(
        location=coord,
        popup=popup,
        icon=folium.Icon(color="green", icon="camera", prefix="fa")
    ).add_to(m)


# Muestra el mapa dentro de Google Colab
display(m)

# Guarda el mapa como archivo HTML
# Este archivo puede abrirse en cualquier navegador
m.save("mapa_completo.html")


Output hidden; open in https://colab.research.google.com to view.